In [1]:
# each input comes with a corresponding correct output in dataset .json file 
# so it is Supervised learning for ASR 
# ex:
# {
#   "audio": "../dataset/chunks/sample_audio_0/0.wav",
#   "text": "अब काफी अच्छा होता है..."
# }

# so for fine-tuning models we will use Whisper


In [2]:
# So, Convert JSON → HuggingFace Dataset bz
# This is just a normal JSON file (list of dictionaries)
# But HuggingFace models don’t train directly on raw JSON
# They need a special format called: "Dataset object (datasets.Dataset)"
# and this loads data efficiently, supports .map() (for preprocessing),and works directly with Trainer


In [3]:
from datasets import load_dataset

dataset_1 = load_dataset("json", data_files="../dataset/dataset.json")
# It converts your JSON into something like:
# dataset["train"][0] 
#output:
# #{
#   "audio": "../dataset/chunks/sample_audio_0/0.wav",
#   "text": "अब काफी अच्छा होता है..."
# }

c:\Users\rishu\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Preprocess data for Whisper (Audio → Model Input)
# Model cannot understand this directly with that .json format so,
# audio file→numerical features (internally: .wav audio → waveform → spectrogram → input_features )
# text→token ids (text → tokenizer → labels)

In [5]:
# so firstly Load Whisper processor + model
from transformers import WhisperProcessor, WhisperForConditionalGeneration

model_name = "openai/whisper-small"

processor = WhisperProcessor.from_pretrained(model_name)
model = WhisperForConditionalGeneration.from_pretrained(model_name)

# 🔥 IMPORTANT for Hindi
processor.tokenizer.set_prefix_tokens(language="hi", task="transcribe")

W0330 03:43:08.174000 24704 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
c:\Users\rishu\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [6]:
# Preprocessing function for Whisper
import librosa

def preprocess(example):
    # Fix path
    audio_path = example["audio"].replace("\\", "/")

    # Load audio
    audio, sr = librosa.load(audio_path, sr=16000)

    # Audio → features
    inputs = processor(
        audio,
        sampling_rate=16000
    )

    # Text → tokens
    labels = processor.tokenizer(example["text"]).input_ids

    return {
        "input_features": inputs.input_features[0],
        "labels": labels
    }

In [7]:
#Apply Preprocessing
dataset_1 = dataset_1["train"]
dataset_1 = dataset_1.map(preprocess)

Map: 100%|██████████| 4259/4259 [01:33<00:00, 45.73 examples/s] 


In [8]:
dataset_1 = dataset_1.remove_columns(["audio", "text"])
# Remove raw columns → keep only model-ready data
# JSON file is still safe ✅

In [10]:
sample = dataset_1[0]

print("Keys:", sample.keys())
print("Feature length:", len(sample["input_features"]))
print("Labels:", sample["labels"][:10])

Keys: dict_keys(['input_features', 'labels'])
Feature length: 80
Labels: [50258, 50276, 50359, 50363, 3941, 227, 3941, 105, 31970, 17937]


In [11]:
# now dataset contains ONLY:
#     input_features → audio converted to numbers , Feature length: 80(Feature length: 80)
#     labels → text converted to tokens
# These token IDs of your Hindi sentence

In [16]:
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    processor.tokenizer,
    model=model
)

training_args = TrainingArguments(
    output_dir="./whisper-finetuned",
    per_device_train_batch_size=4,
    learning_rate=1e-5,
    num_train_epochs=3,
    fp16=False,
    logging_steps=10,
    save_steps=500,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset_1,
    data_collator=data_collator   # ✅ FIX
)

trainer.train()

ValueError: You should supply an encoding or a list of encodings to this method that includes input_ids, but you provided ['input_features', 'labels']

In [1]:
import json
import os

# Your finutune.ipynb is inside /extract/, so dataset is one level up
DATASET_JSON = "../dataset/dataset.json"
DATASET_ROOT = "../"  # base for resolving relative paths

with open(DATASET_JSON, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

# Fix Windows backslashes → forward slashes + make paths absolute
for item in raw_data:
    # "..\\dataset\\chunks\\sample_audio_0\\0.wav" → proper OS path
    item["audio"] = os.path.normpath(
        os.path.join(os.path.dirname(DATASET_JSON), item["audio"].replace("\\\\", "\\"))
    )

print(f"✅ Total samples: {len(raw_data)}")
print(f"📁 Example path: {raw_data[0]['audio']}")
print(f"📝 Example text: {raw_data[0]['text']}")

✅ Total samples: 4259
📁 Example path: ..\dataset\chunks\sample_audio_0\0.wav
📝 Example text: अब काफी अच्छा होता है क्योंकि उनकी जनसंख्या बहुत कम दी जा रही है तो हमें उनको देखना था तो एक देखना था मतलब वो तो देखना था लेकिन हमारा प्रोजेक्ट भी था कि जो जन जाती पाई जाती है उधर कि उधर की एरिया में उसके बारे में देखना अब


In [20]:
import json
import os

DATASET_JSON = "../dataset/dataset.json"

with open(DATASET_JSON, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

# Fix Windows backslash paths
for item in raw_data:
    item["audio"] = os.path.normpath(
        os.path.join(os.path.dirname(DATASET_JSON), item["audio"].replace("\\\\", "\\"))
    )

# ✅ Just load as plain dict — NO cast_column(Audio(...))
from datasets import Dataset

dataset = Dataset.from_list(raw_data)
dataset = dataset.train_test_split(test_size=0.1, seed=42)

print(dataset)
print("✅ Sample path:", raw_data[0]["audio"])

DatasetDict({
    train: Dataset({
        features: ['audio', 'text'],
        num_rows: 3833
    })
    test: Dataset({
        features: ['audio', 'text'],
        num_rows: 426
    })
})
✅ Sample path: ..\dataset\chunks\sample_audio_0\0.wav


In [21]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration

MODEL_NAME = "openai/whisper-small"

processor = WhisperProcessor.from_pretrained(
    MODEL_NAME, language="Hindi", task="transcribe"
)
model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)

# ✅ Fix 1: force Hindi transcription (prevents language detection warning)
model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="hi",
    task="transcribe"
)
model.config.suppress_tokens = []

# ✅ Fix 2: also set generation config (prevents the step-500 warning)
model.generation_config.language = "hi"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="hi",
    task="transcribe"
)

print("✅ Model loaded with Hindi forced!")
print("GPU available:", torch.cuda.is_available())

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


✅ Model loaded with Hindi forced!
GPU available: True


In [22]:
import librosa

def prepare_dataset(batch):
    audio_path = batch["audio"]  # now it's just a string path

    # ✅ Load with librosa — no torchcodec/FFmpeg needed
    audio_array, sampling_rate = librosa.load(audio_path, sr=16000)

    # Extract mel spectrogram features
    batch["input_features"] = processor.feature_extractor(
        audio_array,
        sampling_rate=16000
    ).input_features[0]

    # Tokenize Hindi text
    batch["labels"] = processor.tokenizer(batch["text"]).input_ids

    return batch

dataset = dataset.map(
    prepare_dataset,
    remove_columns=["audio", "text"],  # explicitly remove these two columns
    num_proc=1
)

print("✅ Dataset preprocessed!")
print(dataset)















































































































































































































































Map: 100%|██████████| 3833/3833 [00:36<00:00, 105.20 examples/s]
























Map: 100%|██████████| 426/426 [00:02<00:00, 142.96 examples/s]

✅ Dataset preprocessed!
DatasetDict({
    train: Dataset({
        features: ['input_features', 'labels'],
        num_rows: 3833
    })
    test: Dataset({
        features: ['input_features', 'labels'],
        num_rows: 426
    })
})


In [23]:
# Data Collector

from dataclasses import dataclass
from typing import Any, Dict, List, Union
import torch

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]):
        # Pad input features (mel spectrograms)
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # Pad label sequences
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # Replace padding with -100 so loss ignores it
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )

        # Remove BOS token if prepended
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)
print("✅ Data collator ready!")


✅ Data collator ready!


In [ ]:
!pip install tensorboard

In [ ]:
print("kernel alive")

In [ ]:
#  Metrics + Trainer

import evaluate
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # Replace -100 back to pad token
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str  = processor.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

training_args = Seq2SeqTrainingArguments(
    output_dir="../extract/whisper-finetuned",
    per_device_train_batch_size=4,        # lower to 2 if you get OOM
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,        # effective batch size = 16
    learning_rate=1e-5,
    warmup_steps=200,
    max_steps=3000,
    gradient_checkpointing=True,
    fp16=torch.cuda.is_available(),       # auto-enables on GPU (you have cu118 ✅)
    evaluation_strategy="steps",
    #eval_strategy="steps",
    eval_steps=500,
    save_steps=500,
    save_total_limit=3,                   # keep only last 3 checkpoints to save disk space
    logging_steps=25,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,              # lower WER = better model
    predict_with_generate=False,
    generation_max_length=225,
    report_to="none", 
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
)

print(" Trainer ready! Starting training...")
trainer.train()

In [26]:
import shutil
shutil.rmtree("../extract/whisper-finetuned", ignore_errors=True)
print("✅ Cleared old checkpoints")

✅ Cleared old checkpoints


In [15]:
import os
checkpoints = os.listdir("../extract/whisper-finetuned/")
print(checkpoints)  # should show ['checkpoint-500']

[]
